# Layer 2: qualitative review

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path("/workspace")
CROSSFARM_DIR = PROJECT_ROOT / "output" / "14_layer2_crossfarm_eval"
DATASET_DIR = PROJECT_ROOT / "output" / "09_layer2_dataset"
LABEL_DIR = PROJECT_ROOT / "output" / "11_layer2_proxy_labels"
DICT_DIR = PROJECT_ROOT / "output" / "06_dictionary_refined"
AUDIT_DIR = PROJECT_ROOT / "output" / "07_audits_refined"
OUT_DIR = PROJECT_ROOT / "output" / "16_layer2_qualitative"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = LOG_DIR / f"14_layer2_qualitative_review_fixed_{RUN_TAG}.log"

def log(msg: str):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_json_dump(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

log("Notebook started.")

In [ ]:
pred_path = CROSSFARM_DIR / "crossfarm_fold_predictions.csv.gz"
metrics_path = CROSSFARM_DIR / "crossfarm_metrics_summary.csv"
master_path = DATASET_DIR / "layer2_master_table.csv.gz"
labels_path = LABEL_DIR / "proxy_labels_master.csv.gz"

pred_df = read_csv_auto(pred_path)
metrics_df = read_csv_auto(metrics_path)
master_df = read_csv_auto(master_path)
labels_df = read_csv_auto(labels_path)

log(f"Loaded predictions: {pred_df.shape}")
log(f"Loaded master_df: {master_df.shape}")
log(f"Loaded labels_df: {labels_df.shape}")

display(pred_df.head(2))


In [ ]:
optional_files = {
    "token_events": DATASET_DIR / "layer2_token_events.csv.gz",
    "token_lexicon": DATASET_DIR / "layer2_token_lexicon.csv",
    "unit_inventory": DICT_DIR / "unit_inventory.csv",
    "unit_prototypes": DICT_DIR / "unit_prototypes.csv",
    "unit_farm_bias": AUDIT_DIR / "unit_farm_bias.csv",
    "unit_microphone_bias": AUDIT_DIR / "unit_microphone_bias.csv",
    "unit_placement_bias": AUDIT_DIR / "unit_placement_bias.csv",
    "unit_zone_bias": AUDIT_DIR / "unit_zone_bias.csv",
}
loaded_optional = {}
for name, path in optional_files.items():
    if path.exists():
        loaded_optional[name] = read_csv_auto(path)
        log(f"Loaded {name}: {loaded_optional[name].shape}")
    else:
        log(f"Optional file missing, skipping: {path}")

token_events_df = loaded_optional.get("token_events")
token_lexicon_df = loaded_optional.get("token_lexicon")
unit_inventory_df = loaded_optional.get("unit_inventory")

In [ ]:
review_df = pred_df.merge(master_df, on="_record_key", how="left", suffixes=("", "__master"))
review_df = review_df.merge(labels_df, on="_record_key", how="left", suffixes=("", "__label"))

proba_cols = [c for c in review_df.columns if c.startswith("proba__")]
if proba_cols:
    review_df["confidence_max"] = review_df[proba_cols].max(axis=1)
elif "score__decision" in review_df.columns:
    review_df["confidence_max"] = 1 / (1 + np.exp(-np.abs(review_df["score__decision"].astype(float))))
else:
    review_df["confidence_max"] = np.nan

review_df["is_correct"] = (review_df["y_true"] == review_df["y_pred"]).astype(int)
log(f"Unified review frame shape={review_df.shape}")

In [ ]:
target_to_labelcol = {
    "proxy_functional_v1": "proxy_functional_v1",
    "is_nonvocal": None,
    "is_distress_related": None,
    "is_resource_related": None,
}
target_label_maps = {}
for target, label_col in target_to_labelcol.items():
    if label_col is None:
        target_label_maps[target] = {0: "0", 1: "1"}
    else:
        vals = (
            review_df.loc[review_df["target_name"] == target, label_col]
            .dropna().astype(str).sort_values().unique().tolist()
        )
        target_label_maps[target] = {i: v for i, v in enumerate(vals)}

def decode_label(target, value):
    try:
        return target_label_maps[target].get(int(value), str(value))
    except Exception:
        return str(value)

review_df["y_true_label"] = [decode_label(t, y) for t, y in zip(review_df["target_name"], review_df["y_true"])]
review_df["y_pred_label"] = [decode_label(t, y) for t, y in zip(review_df["target_name"], review_df["y_pred"])]

In [ ]:
# The cross-farm predictions already hold the per-fold selected bundle for each
# target, so the qualitative review runs directly on the logistic-regression rows.
best_pred_df = review_df[review_df["model_name"] == "logreg_balanced"].copy()

best_df = (
    metrics_df[metrics_df["model_name"] == "logreg_balanced"]
    [["target_name", "macro_f1", "balanced_accuracy"]]
    .sort_values("target_name")
    .reset_index(drop=True)
)
display(best_df)
display(best_pred_df.groupby("target_name").size().rename("n_rows"))
log(f"Rows in review frame={len(best_pred_df)}")


In [ ]:
target_summary_rows = []
for (target, model_name), grp in best_pred_df.groupby(["target_name", "model_name"]):
    target_summary_rows.append({
        "target_name": target,
        "model_name": model_name,
        "n_rows": int(len(grp)),
        "n_correct": int(grp["is_correct"].sum()),
        "accuracy": float(grp["is_correct"].mean()),
        "confidence_mean": None if grp["confidence_max"].isna().all() else float(grp["confidence_max"].mean()),
        "confidence_correct_mean": None if grp.loc[grp["is_correct"] == 1, "confidence_max"].isna().all() else float(grp.loc[grp["is_correct"] == 1, "confidence_max"].mean()),
        "confidence_error_mean": None if grp.loc[grp["is_correct"] == 0, "confidence_max"].isna().all() else float(grp.loc[grp["is_correct"] == 0, "confidence_max"].mean()),
    })
target_summary_df = pd.DataFrame(target_summary_rows).sort_values("target_name").reset_index(drop=True)

farm_summary_rows = []
for (target, heldout_farm), grp in best_pred_df.groupby(["target_name", "heldout_farm"]):
    farm_summary_rows.append({
        "target_name": target,
        "heldout_farm": heldout_farm,
        "n_rows": int(len(grp)),
        "accuracy": float(grp["is_correct"].mean()),
        "mean_confidence": None if grp["confidence_max"].isna().all() else float(grp["confidence_max"].mean()),
    })
farm_summary_df = pd.DataFrame(farm_summary_rows).sort_values(["target_name", "heldout_farm"]).reset_index(drop=True)

display(target_summary_df)
display(farm_summary_df)


In [ ]:
def select_cases(df: pd.DataFrame, top_k: int = 12):
    keep_cols = [
        "_record_key", "target_name", "bundle_name", "model_name", "variant", "audio_feature_name",
        "heldout_farm", "confidence_max", "is_correct",
        "y_true", "y_pred", "y_true_label", "y_pred_label",
        "clip_id", "audio_path", "farm", "barn_zone", "microphone", "mic_placement_context",
        "main_category_final", "sub_category_final", "proxy_functional_v1",
        "sequence_unit_ids", "n_segments", "count_high_flagged_unique_tokens",
        "has_any_bias_flag_token", "has_any_high_bias_flag_token"
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    high_correct = df[df["is_correct"] == 1].sort_values(["confidence_max"], ascending=[False]).head(top_k)[keep_cols].copy()
    high_wrong = df[df["is_correct"] == 0].sort_values(["confidence_max"], ascending=[False]).head(top_k)[keep_cols].copy()
    return high_correct, high_wrong

correct_catalogs, wrong_catalogs = [], []
for target, grp in best_pred_df.groupby("target_name"):
    c_ok, c_bad = select_cases(grp, top_k=12)
    correct_catalogs.append(c_ok)
    wrong_catalogs.append(c_bad)

high_conf_correct_df = pd.concat(correct_catalogs, ignore_index=True) if correct_catalogs else pd.DataFrame()
high_conf_wrong_df = pd.concat(wrong_catalogs, ignore_index=True) if wrong_catalogs else pd.DataFrame()

display(high_conf_correct_df.head(10))
display(high_conf_wrong_df.head(10))


In [ ]:
confusion_rows = []
for (target, true_lab, pred_lab), grp in best_pred_df.groupby(["target_name", "y_true_label", "y_pred_label"]):
    if true_lab == pred_lab:
        continue
    confusion_rows.append({
        "target_name": target,
        "true_label": true_lab,
        "pred_label": pred_lab,
        "n_errors": int(len(grp)),
        "mean_confidence": None if grp["confidence_max"].isna().all() else float(grp["confidence_max"].mean()),
        "dominant_farm": grp["heldout_farm"].astype(str).mode().iloc[0] if len(grp) else None,
    })
confusion_patterns_df = pd.DataFrame(confusion_rows).sort_values(["target_name","n_errors"], ascending=[True,False]).reset_index(drop=True)
display(confusion_patterns_df.head(20))

In [ ]:
token_signature_rows = []
if token_events_df is not None and "unit_id" in token_events_df.columns:
    token_base = token_events_df[["_record_key", "unit_id"]].copy()
    for target in best_df["target_name"].tolist():
        sub = best_pred_df[best_pred_df["target_name"] == target].copy()
        rec_correct = sub.loc[sub["is_correct"] == 1, "_record_key"].unique().tolist()
        rec_wrong = sub.loc[sub["is_correct"] == 0, "_record_key"].unique().tolist()
        tok_correct = token_base[token_base["_record_key"].isin(rec_correct)]["unit_id"].value_counts(normalize=True)
        tok_wrong = token_base[token_base["_record_key"].isin(rec_wrong)]["unit_id"].value_counts(normalize=True)
        all_units = sorted(set(tok_correct.index.tolist()) | set(tok_wrong.index.tolist()))
        for unit in all_units:
            token_signature_rows.append({
                "target_name": target,
                "unit_id": unit,
                "freq_correct": float(tok_correct.get(unit, 0.0)),
                "freq_wrong": float(tok_wrong.get(unit, 0.0)),
                "delta_wrong_minus_correct": float(tok_wrong.get(unit, 0.0) - tok_correct.get(unit, 0.0)),
            })

token_signature_df = pd.DataFrame(token_signature_rows)
if not token_signature_df.empty:
    token_signature_df = token_signature_df.sort_values(["target_name","delta_wrong_minus_correct"], ascending=[True,False]).reset_index(drop=True)
    if token_lexicon_df is not None and "unit_id" in token_lexicon_df.columns:
        token_signature_df = token_signature_df.merge(token_lexicon_df.drop_duplicates(subset=["unit_id"]), on="unit_id", how="left")
    elif unit_inventory_df is not None and "unit_id" in unit_inventory_df.columns:
        token_signature_df = token_signature_df.merge(unit_inventory_df.drop_duplicates(subset=["unit_id"]), on="unit_id", how="left")

display(token_signature_df.head(20) if not token_signature_df.empty else pd.DataFrame())


In [ ]:
target_summary_path = OUT_DIR / "qualitative_target_summary.csv"
farm_summary_path = OUT_DIR / "qualitative_farm_summary.csv"
correct_path = OUT_DIR / "high_confidence_correct_cases.csv"
wrong_path = OUT_DIR / "high_confidence_wrong_cases.csv"
confusion_path = OUT_DIR / "confusion_patterns_summary.csv"
token_sig_path = OUT_DIR / "token_error_signature_summary.csv"
manifest_path = OUT_DIR / "qualitative_review_manifest.json"

target_summary_df.to_csv(target_summary_path, index=False)
farm_summary_df.to_csv(farm_summary_path, index=False)
high_conf_correct_df.to_csv(correct_path, index=False)
high_conf_wrong_df.to_csv(wrong_path, index=False)
confusion_patterns_df.to_csv(confusion_path, index=False)
if not token_signature_df.empty:
    token_signature_df.to_csv(token_sig_path, index=False)

manifest = {
    "build_ts_utc": RUN_TS,
    "crossfarm_predictions_source": str(pred_path),
    "crossfarm_metrics_source": str(metrics_path),
    "master_source": str(master_path),
    "labels_source": str(labels_path),
    "n_prediction_rows": int(len(pred_df)),
    "n_review_rows": int(len(best_pred_df)),
    "model_restricted_to": "logreg_balanced",
    "targets_reviewed": best_df.to_dict(orient="records"),
    "outputs": {
        "target_summary_csv": target_summary_path.name,
        "farm_summary_csv": farm_summary_path.name,
        "high_confidence_correct_cases_csv": correct_path.name,
        "high_confidence_wrong_cases_csv": wrong_path.name,
        "confusion_patterns_summary_csv": confusion_path.name,
        "token_error_signature_summary_csv": token_sig_path.name if not token_signature_df.empty else None,
    }
}
safe_json_dump(manifest, manifest_path)
log("Notebook completed.")
